In [1]:
import pandas as pd
import string, json, re

def to_float(value):
    """
    Converts a value to float. 
    Returns 0.0 if the value is None, empty, or not a valid number.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    try:
        clean_val = str(value).replace(',', '').strip()
        return float(clean_val)
    except (ValueError, TypeError):
        return None

def to_int(value):
    """
    Converts a value to int. 
    Returns 0 if the value is None, empty, or not a valid number.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    try:
        clean_val = str(value).replace(',', '').strip()
        return int(clean_val)
    except (ValueError, TypeError):
        return None
    
def to_bool(value):
    """
    Converts a value to boolean.
    Returns None if the value is None or empty.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    str_val = str(value).strip().lower()
    if str_val in ['true', '1', 'yes']:
        return True
    elif str_val in ['false', '0', 'no']:
        return False
    else:
        return None

def process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns):
    """
    Flattens all columns in the dataframe into a simple key:value JSON structure.
    """
    df = df.rename(columns=rename_columns)
    exclude_from_data = ["workorder_id", "filename"] + exclude_cols

    for _, row in df.iterrows():
        wo_id = str(row.get("workorder_id", ""))
        fname = row.get("filename", "unknown")
        
        if not wo_id or wo_id == "nan":
            continue

        if wo_id not in final_json:
            final_json[wo_id] = {
                "filename": fname,
                "data": {}
            }

        row_flattened_data = {}
        for col in df.columns:
            if col in exclude_from_data:
                continue
            
            value = row[col]
            
            if any(num_col in col for num_col in float_columns):
                row_flattened_data[col] = to_float(value)
            elif any(num_col in col for num_col in int_columns):
                row_flattened_data[col] = to_int(value)
            elif any(bool_col in col for bool_col in bool_columns):
                row_flattened_data[col] = to_bool(value)
            else:
                row_flattened_data[col] = value if pd.notna(value) else None
                
        final_json[wo_id]["data"].update(row_flattened_data)

    return final_json

### Air Compressor

In [ ]:
# path = "../../output/tnm/air_compressor.xlsx" 

# df = pd.read_excel(path, sheet_name="air_compressor", keep_default_na=False)
# final_json = {}

# rename_columns = {}
# exclude_cols = []
# float_columns = []
# int_columns = []
# bool_columns = []

# process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

# rows = []

# for workorder_no, payload in final_json.items():
#     rows.append({
#         "workorder_no": workorder_no,
#         "filename": payload.get("filename"),
#         "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
#     })

# df_out = pd.DataFrame(rows)

# output_file = f"../../output/tnm/response_tpss_inspection_weekly.xlsx"
# df_out.to_excel(output_file, index=False)

# print(f"Saved as: {output_file}")


### Hydraulic Press

In [ ]:
path = "../../output/tnm/hydraulic_press.xlsx" 

df = pd.read_excel(path, sheet_name="hydraulic_press", keep_default_na=False)

rename_columns = {}
for col in df.columns:
    match = re.match(r"^(frame|hydraulic)_([a-z])_", col)
    if match:
        section, letter = match.groups()
        rename_columns[col] = f"{section}.{letter}.status"

df = df.rename(columns=rename_columns)



final_json = {}
rename_columns = {}
exclude_cols = []
float_columns = []
int_columns = []
bool_columns = [
    'frame.a.status', 'frame.b.status', 'frame.c.status', 
    'frame.d.status', 'frame.e.status', 'frame.f.status',
    'frame.g.status', 'hydraulic.a.status', 'hydraulic.b.status',
    'hydraulic.c.status', 'hydraulic.d.status', 'hydraulic.e.status',
    'hydraulic.f.status'
]

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_hydraulic_press.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")


### Padestal Grinder

In [ ]:
path = "../../output/tnm/padestal_grinder.xlsx" 

df = pd.read_excel(path, sheet_name="padestal_grinder", keep_default_na=False)

rename_columns = {}
for col in df.columns:
    match = re.match(r"^(grinding_wheel|pedestal_frame|work_rest|safety_glass|electric_motor|water_pot)_([a-z])_", col)
    if match:
        section, letter = match.groups()
        rename_columns[col] = f"{section}.{letter}.status"

df = df.rename(columns=rename_columns)

final_json = {}
rename_columns = {
    "technician_technician_id" : "technician_id",
    "guideway_supervisor_supervisor_id" : "supervisor_id"
}
exclude_cols = []
float_columns = []
int_columns = []
bool_columns = [
    'grinding_wheel.a.status',
    'grinding_wheel.b.status', 'grinding_wheel.c.status',
    'pedestal_frame.a.status', 'work_rest.a.status',
    'safety_glass.a.status', 'electric_motor.a.status',
    'electric_motor.b.status', 'electric_motor.c.status',
    'water_pot.a.status', 'water_pot.b.status'
]

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_padestal_grinder.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")


### Traverser Lubrication

In [ ]:
path = "../../output/tnm/traverser_lubrication.xlsx" 

df = pd.read_excel(path, sheet_name="traverser_lubrication", keep_default_na=False)

final_json = {}
       
rename_columns = {
    "a_carriage_axle_bearings_ep_2_grease_5_strokes_with_hand_grease_gun_each_service" : "carriage_axle_bearings.status",
    "b_locking_pins_ep_2_grease_3_strokes_with_hand_grease_gun_each_service" : "locking_pins.status",
    "c_car_stops_ep_2_grease_3_strokes_with_hand_grease_gun_each_service" : "car_stops.status",
    "d_exposed_drive_and_driven_gears_open_gear_spray_compound_spray_a_full_coating_on_each_gear_each_service" : "exposed_drive_and_driven_gears.status",
    "e_drive_gearbox_220_grade_oil_as_required_each_service" : "drive_gearbox.status",
    "f_hydraulic_reservoir_68_series_oil_as_required_each_service" : "hydraulic_reservoir.status",
    "g_drive_axle_bearings_ep_2_grease_5_strokes_with_hand_grease_gun_each_service" : "drive_axle_bearings.status",
    "technician_technician_id" : "technician_id",
    "guideway_supervisor_supervisor_id" : "supervisor_id"
}
exclude_cols = []
float_columns = []
int_columns = []
bool_columns = [
    'carriage_axle_bearings.status', 'locking_pins.status', 
    'car_stops.status', 'exposed_drive_and_driven_gears.status', 
    'drive_gearbox.status', 'hydraulic_reservoir.status',
    'drive_axle_bearings.status'
]

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_traverser_lubrication.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")


### Traverser Mechanical

In [ ]:
path = "../../output/tnm/traverser_mechanical.xlsx" 

df = pd.read_excel(path, sheet_name="traverser_mechanical", keep_default_na=False)

rename_columns = {}
for col in df.columns:
    match = re.match(r"^(power_rail|hydraulics|carriage|traverser_beam)_([a-z])_", col)
    if match:
        section, letter = match.groups()
        rename_columns[col] = f"{section}.{letter}.status"

df = df.rename(columns=rename_columns)

df.columns
       
final_json = {}
rename_columns = {
    "technician_technician_id" : "technician_id",
    "guideway_supervisor_supervisor_id" : "supervisor_id"
}

exclude_cols = []
float_columns = []
int_columns = []
bool_columns = [
    'power_rail.a.status', 'power_rail.b.status', 'power_rail.c.status', 
    'power_rail.d.status', 'power_rail.e.status', 'power_rail.f.status', 
    'power_rail.g.status', 'power_rail.h.status', 'hydraulics.a.status', 
    'hydraulics.b.status','hydraulics.c.status', 'hydraulics.d.status', 
    'hydraulics.e.status', 'carriage.a.status', 'carriage.b.status', 
    'carriage.c.status', 'carriage.d.status', 'carriage.e.status', 
    'carriage.f.status', 'carriage.g.status', 'carriage.h.status', 
    'traverser_beam.a.status', 'traverser_beam.b.status', 'traverser_beam.c.status'
]

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_traverser_mechanical.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")


### Overhead Crane

In [ ]:
path = "../../output/tnm/overhead_crane.xlsx" 
df = pd.read_excel(path, sheet_name="overhead_crane", keep_default_na=False)

valid_components = [
    # --- Previous Components ---
    'crab_frame_assembly', 'hoist_motor_assembly', 'hook_and_sheaves', 
    'travel_motor_gearbox', 'pendant', 'rope', 'main_girder', 
    'travel_rail_travel_flange', 'limit_stops', 'limit_switches', 
    'latching_device', 'main_girder_end_carriagebolted_connections', 
    'connections', 'end_carriages', 'isolating_switch', 
    'crane_switch_main_contractor', 'control_unit', 'remote_controls', 'fuse_link',

    # --- Control System (Continued) ---
    'contactors', 'transformers', 'electrical_components', 
    'regulating_control_devices', 'electrical_devices', 
    'hoist_unit_limit_switches_collision_prevention_system_by_pass_control_system_reflectors_photocells',
    'warning_devices', 'drive_regulating_devices_control_system', 
    'time_frequency_monitoring_relays',

    # --- Power Supply System ---
    'main_power_supply_to_the_crane', 'power_supply_to_crab_or_traveling_hoist_or_crab', 
    'mobile_control_system', 'stationary_cables_and_wiring', 
    'warning_and_recommendation_signs',

    # --- Crane Runway ---
    'mains_connection_switch', 'foundations', 
    'supports_beams_travel_rails_flanges', 'rail_splices', 
    'runway_suspension', 'corrosion_protection'
]

rename_columns = {}
valid_components.sort(key=len, reverse=True)
bool_columns = []

for col in df.columns:
    pattern = r"^(overhead_crane|crane_bridge|control_system|power_supply_system|crane_runway)_([a-z])_([a-z])_(.*)"
    match = re.match(pattern, col)
    
    if match:
        category, primary_idx, sub_idx, remainder = match.groups()
        
        if category == "overhead_crane":
            category = "crab_or_traveling_hoist"
        
        detected_comp = None
        for comp in valid_components:
            if remainder.startswith(comp):
                detected_comp = comp
                break
        
        if detected_comp:
            item = remainder[len(detected_comp)+1:]
            
            if detected_comp == "hoist_unit_limit_switches_collision_prevention_system_by_pass_control_system_reflectors_photocells":
                detected_comp = "hoist_unit_limit_switches_collision_prevention_system_bypass_control_system_reflectors_photocells"
            if detected_comp == "crane_switch_main_contractor":
                detected_comp = "crane_switch_main_contactor"
            
            new_name = f"{category}.{detected_comp}.{sub_idx}.status"
            rename_columns[col] = new_name
            bool_columns.append(new_name)

df = df.rename(columns=rename_columns)

# df.columns

final_json = {}
rename_columns = {
    "technician_technicians_id" : "technician_id",
    "guideway_supervisor_supervisor_id" : "supervisor_id"
}

exclude_cols = []
float_columns = []
int_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_overhead_crane.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

### Bogie Drop Pit Mechanical

In [ ]:
path = "../../output/tnm/pit_mechanical.xlsx" 
df = pd.read_excel(path, sheet_name="pit_mechanical", keep_default_na=False)

rename_columns = {}
bool_columns = []

for col in df.columns:
    match = re.match(r"^(support_table|rotating_table|lift_cylinder_and_support_frame|hydraulic|safety)_([a-z])_", col)
    if match:
        section, letter = match.groups()
        rename_columns[col] = f"{section}.{letter}.status"
        bool_columns.append(f"{section}.{letter}.status")

df = df.rename(columns=rename_columns)

df.columns

final_json = {}
rename_columns = {
    "technician_signature" : "technician_id",
    "guideway_supervisor" : "supervisor_id"
}

exclude_cols = []
float_columns = []
int_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_bogiedroppit_mechanical.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

### Train Wash Plant

In [ ]:
path = "../../output/tnm/train_wash_plant.xlsx" 
df = pd.read_excel(path, sheet_name="train_wash_plant", keep_default_na=False)

rename_columns = {}
bool_columns = []

for col in df.columns:
    match = re.match(r"^(unit_action_area_and_motion_rails_trimming|safety_devices|light_barriers|under_chassis_cylinder|water_system_nozzles|trimming|hoses|tilt_safety_device|guide_pulley|wire_rope)_([a-z])_", col)
    if match:
        section, letter = match.groups()
        rename_columns[col] = f"{section}.{letter}.status"
        bool_columns.append(f"{section}.{letter}.status")

df = df.rename(columns=rename_columns)

df.columns

final_json = {}
rename_columns = {
    "technician_technician_id" : "technician_id",
    "supervisor_supervisor_id" : "supervisor_id"
}

exclude_cols = []
float_columns = []
int_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_train_wash_plant.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

### Tyre Changing Machine

In [ ]:
path = "../../output/tnm/tyre_changing_machine.xlsx" 
df = pd.read_excel(path, sheet_name="tyre_changing_machine", keep_default_na=False)

rename_columns = {}
bool_columns = []

for col in df.columns:
    match = re.match(r"^(hydraulic_cylinder|hydraulic_tank|button_and_wiring|solenoid_valve|electric_chain_hoist|roller_conveyor|screw_an_bolt)_([a-z])_", col)
    if match:
        section, letter = match.groups()
        rename_columns[col] = f"{section}.{letter}.status"
        bool_columns.append(f"{section}.{letter}.status")

df = df.rename(columns=rename_columns)

df.columns

final_json = {}
rename_columns = {
    "technician_technician_id" : "technician_id",
    "supervisor_supervisor_id" : "supervisor_id"
}

exclude_cols = []
float_columns = []
int_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_tyre_changing_machine.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

### Centre Lathe

In [ ]:
path = "../../output/tnm/centre_lathe.xlsx" 
df = pd.read_excel(path, sheet_name="centre_lathe", keep_default_na=False)

rename_columns = {}

bool_columns = []
for col in df.columns:
    match = re.match(r"^(vee_belts|bed_lead_screws|saddle|tool_post|chuck|electric_motor|coolant_pump|traveling_steady_fixed_steady|tail_stock)_([a-z])_", col)
    if match:
        section, letter = match.groups()
        rename_columns[col] = f"{section}.{letter}.status"
        bool_columns.append(f"{section}.{letter}.status")
        
final_json = {}
exclude_cols = []
float_columns = []
int_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_centre_lathe.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

### Padestal Drill

In [ ]:
path = "../../output/tnm/padestal_drill.xlsx" 
df = pd.read_excel(path, sheet_name="padestal_drill", keep_default_na=False)

rename_columns = {}

bool_columns = []
for col in df.columns:
    match = re.match(r"^(vee_belts|pedestal_frame|work_rest|chuck|electric_motor|water_pot)_([a-z])_", col)
    if match:
        section, letter = match.groups()
        rename_columns[col] = f"{section}.{letter}.status"
        bool_columns.append(f"{section}.{letter}.status")
    
final_json = {}

rename_columns.update({
    "technician_technician_id" : "technician_id",
    "guideway_supervisor_supervisor_id" : "supervisor_id"
})

exclude_cols = []
float_columns = []
int_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_padestal_drill.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

### Power Rail

In [ ]:
path = "../../output/tnm/power_rail.xlsx" 
df = pd.read_excel(path, sheet_name="power_rail", keep_default_na=False)

bool_columns = []
for col in df.columns:
    match = re.match(r"^(location)_([a-z])_(.+)$", col)
    if match:
        section, letter, rest = match.groups()
        new_col = f"{section}_{letter}.{rest}"

        rename_columns[col] = new_col
        bool_columns.append(new_col)
    
final_json = {}

rename_columns.update({
    "technician_technician_id" : "technician_id",
    "supervisor_supervisor_id" : "supervisor_id"
})

exclude_cols = []
float_columns = []
int_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_power_rail.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

### Train Maintenance Vehicle

In [ ]:
path = "../../output/tnm/train_maintenance_vehicle.xlsx" 
df = pd.read_excel(path, sheet_name="tmv", keep_default_na=False)

bool_columns = []
exclude_cols = []
rename_columns = {}

for col in df.columns:
    if col.endswith("completed"):
        suffix = col.split(".")
        new_col = f"{suffix[0]}.{suffix[2]}.status"
        rename_columns[col] = new_col
        bool_columns.append(new_col)
    elif col.startswith("technician") or col.startswith("supervisor"):
        rename_columns.update({
            "technician.technician_id" : "technician_id",
            "technician.date" : "technician_date",
            "supervisor.supervisor_id" : "supervisor_id",
            "supervisor.date" : "supervisor_date",
        })
    else:
        exclude_cols.append(col)

# bool_columns
final_json = {}

float_columns = []
int_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = "../../output/tnm/response_tmv.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    df_out.to_excel(
        writer,
        sheet_name="tmv",
        index=False
    )

print(f"Sheet 'tmv' inserted into: {output_file}")

In [ ]:
path = "../../output/tnm/train_maintenance_vehicle.xlsx" 
df = pd.read_excel(path, sheet_name="tmv3", keep_default_na=False)

bool_columns = []
exclude_cols = []
rename_columns = {}

for col in df.columns:
    if col.endswith("completed"):
        suffix = col.split(".")
        new_col = f"{suffix[0]}.{suffix[2]}.status"
        rename_columns[col] = new_col
        bool_columns.append(new_col)
    elif col.startswith("technician") or col.startswith("supervisor"):
        rename_columns.update({
            "technician.technician_id" : "technician_id",
            "technician.date" : "technician_date",
            "supervisor.supervisor_id" : "supervisor_id",
            "supervisor.date" : "supervisor_date",
        })
    else:
        exclude_cols.append(col)

# bool_columns
final_json = {}

float_columns = []
int_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = "../../output/tnm/response_tmv.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    df_out.to_excel(
        writer,
        sheet_name="tmv3",
        index=False
    )

print(f"Sheet 'tmv_response' inserted into: {output_file}")

In [ ]:
path = "../../output/tnm/train_maintenance_vehicle.xlsx" 
df = pd.read_excel(path, sheet_name="tmv3_2", keep_default_na=False)

bool_columns = []
exclude_cols = []
rename_columns = {}

for col in df.columns:
    if col.endswith("completed"):
        suffix = col.split(".")
        new_col = f"{suffix[0]}.{suffix[2]}.status"
        rename_columns[col] = new_col
        bool_columns.append(new_col)
    elif col.startswith("technician") or col.startswith("supervisor"):
        rename_columns.update({
            "technician.technician_id" : "technician_id",
            "technician.date" : "technician_date",
            "supervisor.supervisor_id" : "supervisor_id",
            "supervisor.date" : "supervisor_date",
        })
    else:
        exclude_cols.append(col)

# bool_columns
final_json = {}

float_columns = []
int_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = "../../output/tnm/response_tmv.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    df_out.to_excel(
        writer,
        sheet_name="tmv3_2",
        index=False
    )

print(f"Sheet 'tmv_response' inserted into: {output_file}")

In [ ]:
path = "../../output/tnm/train_maintenance_vehicle.xlsx" 
df = pd.read_excel(path, sheet_name="tmv3_3", keep_default_na=False)

bool_columns = []
exclude_cols = []
rename_columns = {}

for col in df.columns:
    if col.endswith("completed"):
        suffix = col.split(".")
        new_col = f"{suffix[0]}.{suffix[2]}.status"
        rename_columns[col] = new_col
        bool_columns.append(new_col)
    elif col.startswith("technician") or col.startswith("supervisor"):
        rename_columns.update({
            "technician.technician_id" : "technician_id",
            "technician.date" : "technician_date",
            "supervisor.supervisor_id" : "supervisor_id",
            "supervisor.date" : "supervisor_date",
        })
    else:
        exclude_cols.append(col)

# bool_columns
final_json = {}

float_columns = []
int_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = "../../output/tnm/response_tmv.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    df_out.to_excel(
        writer,
        sheet_name="tmv3_3",
        index=False
    )

print(f"Sheet 'tmv_response' inserted into: {output_file}")

In [ ]:
path = "../../output/tnm/train_maintenance_vehicle.xlsx" 
df = pd.read_excel(path, sheet_name="tmv3_4", keep_default_na=False)

bool_columns = []
exclude_cols = []
rename_columns = {}

for col in df.columns:
    if col.endswith("completed"):
        suffix = col.split(".")
        new_col = f"{suffix[0]}.{suffix[2]}.status"
        rename_columns[col] = new_col
        bool_columns.append(new_col)
    elif col.startswith("technician") or col.startswith("supervisor"):
        rename_columns.update({
            "technician.technician_id" : "technician_id",
            "technician.date" : "technician_date",
            "supervisor.supervisor_id" : "supervisor_id",
            "supervisor.date" : "supervisor_date",
        })
    else:
        exclude_cols.append(col)

# bool_columns
final_json = {}

float_columns = []
int_columns = []

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = "../../output/tnm/response_tmv.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    df_out.to_excel(
        writer,
        sheet_name="tmv3_4",
        index=False
    )

print(f"Sheet 'tmv_response' inserted into: {output_file}")

### Air Compressor

In [ ]:
path = "../../output/tnm/air_compressor.xlsx" 
df = pd.read_excel(path, sheet_name="air_compressor", keep_default_na=False)

bool_columns = []
float_columns = []
int_columns = []

for col in df.columns:
    if col.endswith("yellow") or col.endswith("red") or col.endswith("oil_tank") or col.endswith("oil_level") or col.endswith("filtering_panel") or col.endswith("filter"):
        bool_columns.append(col)
    elif col.endswith("machine_number") or col.endswith("running_hours"):
        int_columns.append(col)
    elif col.endswith("pressure_bar") or col.endswith("temp"):
        float_columns.append(col)
        
rename_columns = {
    "supervisor.supervisor_id" : "supervisor_id",
    "supervisor.date" : "supervisor_date"
}

final_json = {}
exclude_cols = []
# bool_columns, int_columns, float_columns

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_air_compressor.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

### Bogie Drop Pit Lub

In [ ]:
path = "../../output/tnm/bogie_drop_pit_lub.xlsx" 
df = pd.read_excel(path, sheet_name="bogie_drop_pit_lub", keep_default_na=False)

allowed_prefixes = (
    "any_additional_work_that_requires_planning",
    "technician",
    "supervisor",
    "workorder_id", "filename"
)

bool_columns = []
float_columns = []
int_columns = []
final_json = {}
exclude_cols = []

for col in df.columns:
    if col.endswith(".completed"):
        new_col = col.rsplit(".", 1)[0] + ".status"
        rename_columns[col] = new_col
        bool_columns.append(new_col)

    elif not col.startswith(allowed_prefixes):
        exclude_cols.append(col)

df = df.rename(columns=rename_columns)
        
df = df.rename(columns={
    "technician.technician_id": "technician_id",
    "technician.date": "technician_date",
    "supervisor.supervisor_id": "supervisor_id",
    "supervisor.date": "supervisor_date",
})

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_bogie_drop_pit_lub.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

### Bogie Test Jig

In [ ]:
path = "../../output/tnm/bogie_test_jig.xlsx" 
df = pd.read_excel(path, sheet_name="bogie_test_jig", keep_default_na=False)

allowed_prefixes = (
    "any_additional_work_that_requires_planning",
    "technician",
    "supervisor",
    "workorder_id", "filename"
)

bool_columns = []
float_columns = []
int_columns = []
final_json = {}
exclude_cols = []

for col in df.columns:
    if col.endswith(".completed"):
        new_col = col.rsplit(".", 1)[0] + ".status"
        rename_columns[col] = new_col
        bool_columns.append(new_col)

    elif not col.startswith(allowed_prefixes):
        exclude_cols.append(col)

df = df.rename(columns=rename_columns)
        
df = df.rename(columns={
    "technician.technician_id": "technician_id",
    "technician.date": "technician_date",
    "supervisor.supervisor_id": "supervisor_id",
    "supervisor.date": "supervisor_date",
})

process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []

for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data", {}), ensure_ascii=False)
    })

df_out = pd.DataFrame(rows)

output_file = f"../../output/tnm/response_bogie_test_jig.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")